In [2]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request
import codecs

In [26]:
df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump1.parquet')
df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump2.parquet')
df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump3.parquet')
df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump4.parquet')
df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump5.parquet')
df6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump6.parquet')
df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump7.parquet')

dump1 = df1.to_pandas()
dump2 = df2.to_pandas()
dump3 = df3.to_pandas()
dump4 = df4.to_pandas()
dump5 = df5.to_pandas()
dump6 = df6.to_pandas()
dump7 = df7.to_pandas()

# Load attack dumps
#attack_044h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-044h.parquet").to_pandas()
#attack_080h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-080h.parquet").to_pandas()
#attack_383h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-383h.parquet").to_pandas()
#attack_593h = pq.read_table(r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-masq-593h.parquet").to_pandas()"""


In [27]:
files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw"
fuzz_files = [
    f for f in os.listdir(files_pathname)
    if f.startswith("dump6-fuzz-") and f.endswith(".parquet")
]

fuzz_ids = [f.split('-')[-1].removesuffix('.parquet') for f in fuzz_files]

print(f"Found {len(fuzz_ids)} fuzzing files.")
print("Fuzzing IDs in discovery order:", fuzz_ids)

Found 17 fuzzing files.
Fuzzing IDs in discovery order: ['10', '100', '1000', '1500', '20', '200', '2000', '30', '300', '40', '400', '50', '500', '60', '70', '80', '90']


In [28]:

def hex_to_bytes(h):
    #Convert hex/bytes/string to bytes.
    if h is None or (isinstance(h, float) and np.isnan(h)): 
        return b""
    if isinstance(h, (bytes, bytearray)): 
        return bytes(h)
    s = str(h).strip().replace("0x","").replace(" ","")
    if s == "": 
        return b""
    if len(s) % 2 == 1: 
        s = "0"+s
    try: 
        return bytes.fromhex(s)
    except Exception: 
        print(f"Warning: invalid hex input: {h}")
        return str(h).encode("utf-8", errors="ignore")


def hamming_distance(a: bytes, b: bytes) -> tuple:
    
    
    len_mismatch = (len(a) != len(b))

    # Pad to same length
    max_len = max(len(a), len(b))
    a_padded = a + b'\x00' * (max_len - len(a))
    b_padded = b + b'\x00' * (max_len - len(b))
    
    # Count differences
    #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
    
    distance = 0
    for byte_a, byte_b in zip(a_padded, b_padded):
        distance += bin(byte_a ^ byte_b).count('1')
    return (distance, len_mismatch)
    
    #return (dist, len_mismatch)

In [29]:

def compute_hamming_distances(dumps, out_csv,ranges, process_per_dump=True):
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        
    records = []
    prev_payload = {}  # {arbitration_id: previous_bytes}
    ranges_indexed = ranges.set_index('arbitration_id')
    

    
    for dump_name, df in dumps:
        d = df.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        
        d = d.sort_values("timestamp", kind="mergesort")
        has_label = "label" in d.columns
        
        for _, row in d.iterrows():
            aid = row["arbitration_id"]
            ts = row["timestamp"]
            lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
            should_update = False
            curr_bytes = hex_to_bytes(row["data"])
            
            
            # Get previous payload for this ID
            prev = prev_payload.get(aid)
            
            if prev is not None:
                # Compute Hamming distance
                dist, len_mismatch = hamming_distance(curr_bytes, prev)
                #max_len = max(len(curr_bytes), len(prev))
                #norm_dist = dist / (max_len * 8) if max_len > 0 else 0.0
                
                records.append({
                    "dump": dump_name,
                    "timestamp": ts,
                    "arbitration_id": aid,
                    "payload_len": len(curr_bytes),
                    "ham_dist": dist,
                    #"ham_norm": norm_dist,
                    "len_mismatch": len_mismatch,  
                    "label": lab
                })

                if ranges_indexed is not None and aid in ranges_indexed.index:
                    min_val = ranges_indexed.at[aid, 'min_hamming']
                    max_val = ranges_indexed.at[aid, 'max_hamming']

                if min_val is not None and max_val is not None:
                    if min_val <= dist <= max_val:
                        should_update = True
                    # else: outside range = suspicious, don't update
                    else:
                        # Unknown ID (not in training) - don't update
                        should_update = False
    
                #Update baseline only if within range
                if should_update:
                    prev_payload[aid] = curr_bytes

            else:
                prev_payload[aid] = curr_bytes
                    
        #False for the bening so we can compute the distances using all 7 dumps
        if process_per_dump:
            prev_payload.clear()
    
    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(out_csv, index=False)
    print(f" Saved: {out_csv} (rows={len(out_df):,})")
    
    return out_df

In [30]:

def build_hamming_range_model(hamming_csv, output_csv="artifacts/hamming_ranges_fuzz.csv"):
    """
    Learn [min, max] Hamming distance range per ID.
    
    For each arbitration_id:
    - min_hamming: Minimum distance seen between consecutive payloads
    - max_hamming: Maximum distance seen
    - range_size: max - min (paper's "Hamming range")
    
    Classification (from paper, σ=6):
    - NoRange:     range_size = 0
    - SmallRange:  0 < range_size ≤ 6
    - MidRange:    range_size > 6
    
    Args:
        hamming_csv: Path to computed Hamming distances (benign data)
        output_csv: Where to save learned ranges
        
    Returns:
        DataFrame with learned ranges per ID
    """
    print("="*70)
    print("🔧 Building Hamming Range Model (Paper Method)")
    print("="*70)
    
    df = pd.read_csv(hamming_csv)
    
    # Group by ID and compute min/max
    ranges = (df.groupby('arbitration_id')['ham_dist']
            .agg(['min', 'max', 'count'])
            .reset_index())
    
    ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
    
    # Compute range size
    ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
    
    # Normalized versions (for 8-byte payloads, max=8)
    ranges['min_norm'] = ranges['min_hamming'] / 64.0
    ranges['max_norm'] = ranges['max_hamming'] / 64.0
    ranges['range_norm'] = ranges['range_size'] / 64.0
    
    # Classify IDs (paper uses sigma=6 for byte-level Hamming)
    sigma = 6 * 4
    """def classify(r):
        if r == 0:
            return 'NoRange'
        elif r <= sigma:
            return 'SmallRange'
        else:
            return 'MidRange'
    
    ranges['class'] = ranges['range_size'].apply(classify)
    
    # Expected detection rates 
    def expected_tpr(cls):
        if cls == 'NoRange':
            return 0.98  
        elif cls == 'SmallRange':
            return 0.97  
        else:
            return 0.25  
    
    ranges['expected_tpr'] = ranges['class'].apply(expected_tpr)
    
    # Sort by range size
    ranges = ranges.sort_values('range_size')"""
    
    # Save
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    ranges.to_csv(output_csv, index=False)
    
    # Print summary
    print(f"\nLearned ranges for {len(ranges)} unique IDs")
    print(f"   Saved to: {output_csv}")
    
    """print(f"\n Range Classification (σ={sigma}):")
    class_counts = ranges['class'].value_counts()
    for cls in ['NoRange', 'SmallRange', 'MidRange']:
        if cls in class_counts.index:
            count = class_counts[cls]
            pct = count / len(ranges) * 100
            print(f"   {cls:12s}: {count:3d} IDs ({pct:5.1f}%)")
    
    print(f"\n Range Statistics:")
    print(f"   Mean range:   {ranges['range_size'].mean():.2f} bits")
    print(f"   Median range: {ranges['range_size'].median():.2f} bits")
    print(f"   Max range:    {ranges['range_size'].max():.0f} bits")
    
    # Show examples from each class
    print(f"\n🔍 Examples by Class:")
    for cls in ['NoRange', 'SmallRange', 'MidRange']:
        examples = ranges[ranges['class'] == cls].head(3)
        if len(examples) > 0:
            print(f"\n   {cls}:")
            for _, row in examples.iterrows():
                print(f"      AID 0x{int(row['arbitration_id']):03X}: "
                    f"range [{row['min_hamming']:.0f}, {row['max_hamming']:.0f}] "
                    f"(size={row['range_size']:.0f}, expected_tpr={row['expected_tpr']*100:.0f}%)")"""
    
    print("="*70)
    return ranges


In [31]:

def detect_simple(test_csv, ranges_csv, output_csv="artifacts/detections_simple_fuzz.csv"):
    """
    Simple detection: Flag if distance is OUTSIDE [min, max] range.
    No windowing - direct per-message detection.
    
    Args:
        test_csv: Path to test data with Hamming distances
        ranges_csv: Path to learned ranges
        output_csv: Where to save results
        
    Returns:
        DataFrame with detection results and performance metrics
    """
    
    # Load data
    test = pd.read_csv(test_csv)
    ranges = pd.read_csv(ranges_csv)
    
    # Merge test data with ranges
    test_with_ranges = test.merge(
        ranges[['arbitration_id', 'min_hamming', 'max_hamming', 'range_size']],
        on='arbitration_id',
        how='left'
    )
    
    # Detect: Flag if OUTSIDE range [min, max]
    test_with_ranges['is_anomaly'] = (
        (test_with_ranges['ham_dist'] < test_with_ranges['min_hamming']) |
        (test_with_ranges['ham_dist'] > test_with_ranges['max_hamming'])
    )
    
    """#-----------------
    #-----------------
    #-----------------
    # Check where your FPs are coming from:
    fps = test_with_ranges[
        (test_with_ranges['label'] == 0) & 
        (test_with_ranges['is_anomaly'] == True)
    ]

    print(f"False Positives: {len(fps)}")
    print(f"\nFP Breakdown:")
    print(f"  Length mismatches: {fps['len_mismatch'].sum()}")
    print(f"  Hamming too low: {(fps['ham_dist'] < fps['min_hamming']).sum()}")
    print(f"  Hamming too high: {(fps['ham_dist'] > fps['max_hamming']).sum()}")

    # Check which IDs produce FPs:
    fp_ids = fps.groupby('arbitration_id').size().sort_values(ascending=False)
    print(f"\nTop FP-producing IDs:")
    print(fp_ids.head(10))"""
    
    #-----------------
    #-----------------
    #-----------------
    
    # Count anomalies
    n_anomalies = test_with_ranges['is_anomaly'].sum()
    n_total = len(test_with_ranges)
    
    print(f"\n Detection Summary:")
    print(f"   Total messages:    {n_total:,}")
    print(f"   Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.2f}%)")
    
    # Compute metrics if labels available
    metrics = {}
    if 'label' in test_with_ranges.columns:
        y_true = test_with_ranges['label'].values
        y_pred = test_with_ranges['is_anomaly'].values
        
        TP = ((y_true == 1) & (y_pred == True)).sum()
        FP = ((y_true == 0) & (y_pred == True)).sum()
        TN = ((y_true == 0) & (y_pred == False)).sum()
        FN = ((y_true == 1) & (y_pred == False)).sum()
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        print(f"\n Detection Results:")
        print(f"   TPR: {tpr*100:.2f}% | FPR: {fpr*100:.2f}% | F1: {f1*100:.2f}%")
        print(f"   TP: {TP:,} | FP: {FP:,} | TN: {TN:,} | FN: {FN:,}")
        
        metrics['overall'] = {
            'TP': int(TP), 'FP': int(FP), 'TN': int(TN), 'FN': int(FN),
            'TPR': tpr, 'FPR': fpr, 'Precision': precision, 'F1': f1
        }
        
        """print(f"\n📋 Detection Rate by ID Class (Paper's Expected vs Actual):")
        print(f"   {'Class':<12} {'Expected':<12} {'Actual':<12} {'IDs':<8}")
        print(f"   {'-'*50}")
        
        for cls in ['NoRange', 'SmallRange', 'MidRange']:
            cls_data = test_with_ranges[test_with_ranges['class'] == cls]
            if len(cls_data) > 0 and 'label' in cls_data.columns:
                cls_attacks = cls_data[cls_data['label'] == 1]
                cls_detected = cls_attacks[cls_attacks['is_anomaly']]
                
                if len(cls_attacks) > 0:
                    actual_tpr = len(cls_detected) / len(cls_attacks)
                    expected_tpr = cls_data['expected_tpr'].iloc[0]
                    n_ids = cls_data['arbitration_id'].nunique()
                    
                    print(f"   {cls:12s} {expected_tpr*100:>6.1f}%      {actual_tpr*100:>6.1f}%      {n_ids:>3d}")
        
        
        # Per-ID breakdown
        per_id = []
        for aid in test_with_ranges['arbitration_id'].unique():
            aid_data = test_with_ranges[test_with_ranges['arbitration_id'] == aid]
            
            if 'label' not in aid_data.columns:
                continue
            
            y_t = aid_data['label'].values
            y_p = aid_data['is_anomaly'].values
            
            tp = ((y_t == 1) & (y_p == True)).sum()
            fn = ((y_t == 1) & (y_p == False)).sum()
            tpr_id = tp / (tp + fn) if (tp + fn) > 0 else 0
            
            aid_class = aid_data['class'].iloc[0] if len(aid_data) > 0 else 'Unknown'
            aid_range = aid_data['range_size'].iloc[0] if len(aid_data) > 0 else np.nan
            
            per_id.append({
                'arbitration_id': aid,
                'aid_hex': f"0x{aid:03X}",
                'n_attacks': int(tp + fn),
                'detected': int(tp),
                'missed': int(fn),
                'tpr': tpr_id,
                'class': aid_class,
                'range_size': aid_range
            })
        
        metrics['per_id'] = pd.DataFrame(per_id)"""
    
    # Save results
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    test_with_ranges.to_csv(output_csv, index=False)
    
    return test_with_ranges, metrics


In [ ]:

if __name__ == "__main__":
    print("\n" + "="*70)
    print("🚀 HAMMING DISTANCE IDS - COMPLETE PIPELINE")
    print("="*70)
    
    training_dumps = [("dump6",dump6)]
    testing_dumps = [
        ("dump1", dump1),
        ("dump2", dump2),
        ("dump3", dump3),
        ("dump4", dump4),
        ("dump5", dump5),
        ("dump7", dump7)
    ]
    
    # ========================================================================
    """training_ham = compute_hamming_distances(
    training_dumps, 
    "training/benign_hamming_train.csv",
    process_per_dump=True  # Keep history across dumps
    )
    """
    ranges = build_hamming_range_model(
    "training/benign_hamming_train.csv",
    "training/hamming_ranges_training.csv"
    )
   
    # ========================================================================
    
    # Get all fuzz files
    #fuzz_files = [f for f in os.listdir(files_pathname) if f.startswith("dump6-fuzz-") and f.endswith(".parquet")]
    
    
    #fuzz_ids = sorted([f.replace("dump6-fuzz-", "").replace(".parquet", "") for f in fuzz_files])
    """
    testing_ham = compute_hamming_distances(
    testing_dumps, 
    "testing/benign_hamming_testing.csv",
    process_per_dump=True  # Keep history across dumps
)"""
    
 
    results_summary = []
    for i, (dump_name, dump_df) in enumerate(testing_dumps, 1):  
        
        try:
            # Load attack file
            #attack_file = os.path.join(files_pathname, f"dump6-fuzz-{fuzz_id}.parquet") 
            #attack_df = pq.read_table(attack_file).to_pandas()
            
            #print(f"   Loaded: {len(attack_df):,} messages")
            testing_ham = compute_hamming_distances(
            [(dump_name, dump_df)], 
            f"testing/hamming_testing_dump{i}.csv",
            ranges,
            process_per_dump=True
        )
            
            
            """# Compute Hamming
            attack_dumps = [(f"fuzz_{fuzz_id}", attack_df)]
            attack_ham = compute_hamming_distances(
                attack_dumps,
                f"artifacts/attack_ham_fuzz_{fuzz_id}.csv", 
                process_per_dump=True  # Reset history for attack dump
            )"""
            
            # Detect
            results, metrics = detect_simple(
                f"testing/hamming_testing_dump{i}.csv",  
                "training/hamming_ranges_training.csv",
                f"testing/detections_FPR_dump{i}.csv" 
            )
            
            # Store results
            if 'overall' in metrics:
                results_summary.append({
                    'dump_i': i,  
                    'n_attacks': metrics['overall']['TP'] + metrics['overall']['FN'],
                    'TPR': metrics['overall']['TPR'] * 100,
                    'FPR': metrics['overall']['FPR'] * 100,
                    'Precision': metrics['overall']['Precision'] * 100,
                    'F1': metrics['overall']['F1'] * 100,
                    'TP': metrics['overall']['TP'],
                    'FP': metrics['overall']['FP'],
                    'TN': metrics['overall']['TN'],
                    'FN': metrics['overall']['FN']
                })
                
                print(f"   TPR: {metrics['overall']['TPR']*100:.1f}%, "
                      f"FPR: {metrics['overall']['FPR']*100:.2f}%, "
                      f"F1: {metrics['overall']['F1']*100:.1f}%")
        
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ========================================================================
    # STEP 4: Summary
    # ========================================================================
    summary_df = pd.DataFrame(results_summary)
    summary_df.to_csv("artifacts/all_fuzz_summary.csv", index=False)
    
    print("FUZZING ATTACKS - FINAL SUMMARY")
   
    
    if len(summary_df) > 0:
        print(f"\nOverall Performance:")
        print(f"   Mean TPR:       {summary_df['TPR'].mean():.2f}%")
        print(f"   Median TPR:     {summary_df['TPR'].median():.2f}%")
        print(f"   Min TPR:        {summary_df['TPR'].min():.2f}%")
        print(f"   Max TPR:        {summary_df['TPR'].max():.2f}%")
        print(f"   Mean FPR:       {summary_df['FPR'].mean():.4f}%")
        print(f"   Mean F1:        {summary_df['F1'].mean():.2f}%")
        
        print("\nResults by Fuzzing Intensity:")
        print(summary_df[['n_attacks', 'TPR', 'FPR', 'F1']].to_string(index=False))
        
        # Compare with masquerade
        print("\n" + "="*70)
        print("COMPARISON: Fuzzing vs Masquerade")
        print("="*70)


🚀 HAMMING DISTANCE IDS - COMPLETE PIPELINE
 Saved: testing/hamming_testing_dump1.csv (rows=3,123,723)

 Detection Summary:
   Total messages:    3,123,723
   Flagged anomalies: 13 (0.00%)

 Detection Results:
   TPR: 0.00% | FPR: 0.00% | F1: 0.00%
   TP: 0 | FP: 13 | TN: 3,123,710 | FN: 0
   TPR: 0.0%, FPR: 0.00%, F1: 0.0%
 Saved: testing/hamming_testing_dump2.csv (rows=4,134,438)

 Detection Summary:
   Total messages:    4,134,438
   Flagged anomalies: 14 (0.00%)

 Detection Results:
   TPR: 0.00% | FPR: 0.00% | F1: 0.00%
   TP: 0 | FP: 14 | TN: 4,134,424 | FN: 0
   TPR: 0.0%, FPR: 0.00%, F1: 0.0%
 Saved: testing/hamming_testing_dump3.csv (rows=3,233,691)

 Detection Summary:
   Total messages:    3,233,691
   Flagged anomalies: 7,695 (0.24%)

 Detection Results:
   TPR: 0.00% | FPR: 0.24% | F1: 0.00%
   TP: 0 | FP: 7,695 | TN: 3,225,996 | FN: 0
   TPR: 0.0%, FPR: 0.24%, F1: 0.0%
 Saved: testing/hamming_testing_dump4.csv (rows=4,761,263)

 Detection Summary:
   Total messages:    4,

KeyError: "['fuzz_id'] not in index"

In [1]:
# On e.g. testing/hamming_testing_dump6.csv (where FP is 0.92%)
test6 = pd.read_csv("testing/hamming_testing_dump6.csv")

# Join ranges
ranges = pd.read_csv("artifacts/hamming_ranges_clean.csv")
df6 = test6.merge(ranges[['arbitration_id','min_hamming','max_hamming']],
                  on='arbitration_id', how='left')

df6['is_anom'] = (df6['ham_dist'] < df6['min_hamming']) | (df6['ham_dist'] > df6['max_hamming'])

print(df6['is_anom'].mean(), "overall anomaly rate")

print("\nTop FPs by AID:")
print(df6[df6['is_anom']].groupby('arbitration_id').size().sort_values(ascending=False).head(10))


NameError: name 'pd' is not defined